# Notebook 01: Extracción e Inventario
## Proyecto II Parcial — Modelado Avanzado de Base de Datos - 30759
## Integrantes:
- Naomi Obando
- Mauri Tandazo
**Fuente:** NYC TLC Trip Record Data + Apache Parquet Testing (bad files)  
**Capa destino:** `data/raw/` → Inventario técnico `audit_file_inventory`

---
### Objetivo de esta fase
Descargar, organizar y leer todos los archivos Parquet, construyendo un inventario técnico
que registre el estado de cada archivo. Los archivos problemáticos son aislados **sin detener el pipeline**.

In [1]:
# ── 0. Setup inicial ──────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.utils   import generate_process_id, load_config, setup_logger, create_spark_session, ensure_dirs
from src.extract import run_extraction_phase, collect_parquet_files, detect_service_type

# Generar ID único de ejecución
PROCESS_ID  = generate_process_id()
CONFIG_PATH = '../config/etl_config.yaml'

print(f'process_id : {PROCESS_ID}')
print(f'config     : {CONFIG_PATH}')

process_id : ETL-20260623-125906-A58E4897
config     : ../config/etl_config.yaml


In [2]:
# ── 1. Cargar configuración y crear directorios ───────────────────
config = load_config(CONFIG_PATH)
logger = setup_logger('nb01_extraccion', '../logs', PROCESS_ID)
ensure_dirs({**config, 'paths': {k: f'../{v}' for k, v in config['paths'].items()}})

# Ajustar rutas para ejecutar desde notebooks/
config['paths'] = {k: f'../{v}' for k, v in config['paths'].items()}
config['paths']['metadata_dir'] = '../metadata'

print('Configuración cargada OK')
print(f"Base de datos: {config['database']['path']}")

Configuración cargada OK
Base de datos: data/warehouse/nyc_tlc.duckdb


In [3]:
# ── 2. Crear SparkSession con optimizaciones ─────────────────────
spark = create_spark_session(config)
print(f'Spark versión : {spark.version}')
print(f'Master        : {spark.sparkContext.master}')
print(f'App Name      : {spark.sparkContext.appName}')

Spark versión : 3.5.0
Master        : local[4]
App Name      : ETL_NYC_TLC_Parquet_Advanced


In [4]:
# ── 3. Ejecutar Fase 1: Extracción completa ───────────────────────
# Esta función:
#   a) Descarga archivos con caché local (idempotente)
#   b) Lee cada .parquet individualmente con manejo de errores
#   c) Registra inventario técnico de cada archivo
#   d) Envía archivos problemáticos a cuarentena

dataframes_by_service, inventory_records = run_extraction_phase(
    spark, config, PROCESS_ID, logger
)

print(f'\nDataFrames yellow : {len(dataframes_by_service["yellow"])}')
print(f'DataFrames green  : {len(dataframes_by_service["green"])}')
print(f'DataFrames fhvhv  : {len(dataframes_by_service["fhvhv"])}')
print(f'Archivos inventariados: {len(inventory_records)}')

2026-06-23 12:59:26 | INFO     | nb01_extraccion                | ══════════════════════════════════════════════════════════════════════
2026-06-23 12:59:26 | INFO     | nb01_extraccion                |   FASE 1: EXTRACCIÓN  —  process_id: ETL-20260623-125906-A58E4897
2026-06-23 12:59:26 | INFO     | nb01_extraccion                | ══════════════════════════════════════════════════════════════════════
2026-06-23 12:59:26 | INFO     | nb01_extraccion                | 
[1.1] Descarga de archivos fuente:
2026-06-23 12:59:26 | INFO     | nb01_extraccion                | ── Descargando archivos NYC TLC ───────────────────────────────
2026-06-23 12:59:26 | INFO     | nb01_extraccion                |   Servicio: YELLOW
2026-06-23 12:59:26 | INFO     | nb01_extraccion                |     [CACHE] yellow_tripdata_2023-01.parquet (45.46 MB) — ya existe, omitiendo descarga
2026-06-23 12:59:26 | INFO     | nb01_extraccion                |     [CACHE] yellow_tripdata_2023-02.parquet (45.54 MB) — y

In [5]:
# ── 4. Visualizar inventario técnico ─────────────────────────────
import pandas as pd
pd.set_option('display.max_colwidth', 60)

inv_df = pd.DataFrame(inventory_records)

cols_show = ['file_name','service_type','file_size_mb',
             'read_status','record_count','column_count',
             'partition_year','partition_month']

def color_status(v):
    # Compatible con pandas >= 2.0 (.map reemplaza .map en Styler)
    if v == 'SUCCESS':
        return 'background-color: #d4edda; color: #155724'
    if isinstance(v, str) and 'NOT_RECOVERABLE' in v:
        return 'background-color: #f8d7da; color: #721c24'
    if isinstance(v, str) and 'RECOVERABLE' in v:
        return 'background-color: #fff3cd; color: #856404'
    return ''

# .map() es el nombre correcto desde pandas 2.0
display(
    inv_df[cols_show].style
    .map(color_status, subset=['read_status'])
    .format({'file_size_mb': '{:.2f}', 'record_count': '{:,}'})
    .set_caption('Inventario Técnico de Archivos Parquet')
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size','14px'),('font-weight','bold'),('color','#1F3864')]
    }])
)


,file_name,service_type,file_size_mb,read_status,record_count,column_count,partition_year,partition_month
0,ARROW-GH-41317.parquet,bad_parquet,0.07,RECOVERABLE_SCHEMA_MISMATCH,0,0,nan,nan
1,ARROW-GH-41321.parquet,bad_parquet,0.07,RECOVERABLE_SCHEMA_MISMATCH,0,0,nan,nan
2,ARROW-GH-43605.parquet,bad_parquet,0.00,SUCCESS,"21,186",1,nan,nan
3,ARROW-GH-45185.parquet,bad_parquet,0.00,SUCCESS,5,1,nan,nan
4,ARROW-GH-47662.parquet,bad_parquet,0.00,SUCCESS,"1,000",1,nan,nan
5,ARROW-RS-GH-6229-DICTHEADER.parquet,bad_parquet,0.00,SUCCESS,0,4,6229,nan
6,ARROW-RS-GH-6229-LEVELS.parquet,bad_parquet,0.00,SUCCESS,1,1,6229,nan
7,PARQUET-1481.parquet,bad_parquet,0.00,NOT_RECOVERABLE_UNSUPPORTED_FORMAT,0,0,nan,nan
8,fhvhv_tripdata_2023-01.parquet,fhvhv,451.87,SUCCESS,"18,479,031",24,2023,01
9,green_tripdata_2023-01.parquet,green,1.36,SUCCESS,"68,211",20,2023,01


In [6]:
# ── 5. Resumen por estado de lectura ─────────────────────────────
print('\n=== RESUMEN POR ESTADO DE LECTURA ===')
summary = inv_df.groupby('read_status').agg(
    archivos=('file_name', 'count'),
    total_registros=('record_count', 'sum'),
    tamano_mb=('file_size_mb', 'sum')
).reset_index()
display(summary)

print('\n=== ARCHIVOS PROBLEMÁTICOS (si existen) ===')
problematic = inv_df[inv_df['read_status'] != 'SUCCESS']
if len(problematic) > 0:
    display(problematic[['file_name','read_status','error_message']].fillna('-'))
else:
    print('Todos los archivos se leyeron correctamente.')


=== RESUMEN POR ESTADO DE LECTURA ===


,read_status,archivos,total_registros,tamano_mb
0,NOT_RECOVERABLE_UNSUPPORTED_FORMAT,1,0,0.00
1,RECOVERABLE_SCHEMA_MISMATCH,2,0,0.14
2,SUCCESS,16,44525119,840.71



=== ARCHIVOS PROBLEMÁTICOS (si existen) ===


,file_name,read_status,error_message
0,ARROW-GH-41317.parquet,RECOVERABLE_SCHEMA_MISMATCH,An error occurred while calling o73.parquet.\n: org.apac...
1,ARROW-GH-41321.parquet,RECOVERABLE_SCHEMA_MISMATCH,An error occurred while calling o82.parquet.\n: org.apac...
7,PARQUET-1481.parquet,NOT_RECOVERABLE_UNSUPPORTED_FORMAT,An error occurred while calling o139.count.\n: org.apach...


In [7]:
# ── 6. Inspección de esquemas por servicio ────────────────────────
print('=== ESQUEMAS POR SERVICIO ===')
for service, files in dataframes_by_service.items():
    if not files:
        continue
    path, df = files[0]
    print(f'\n{service.upper()} — {len(files)} archivos — primer archivo: {path.split("/")[-1]}')
    print(f'  Columnas ({len(df.columns)}): {df.columns}')
    df.printSchema()

=== ESQUEMAS POR SERVICIO ===

YELLOW — 8 archivos — primer archivo: raw\yellow\year=2020\month=01\yellow_tripdata_2020-01.parquet
  Columnas (20): ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', '_source_file']
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = 

In [8]:
# ── 7. Persistir inventario como checkpoint ───────────────────────
import json
inv_path = f'../data/audit/inventory_nb01_{PROCESS_ID}.json'
with open(inv_path, 'w') as f:
    json.dump(inventory_records, f, ensure_ascii=False, indent=2, default=str)

print(f'Inventario guardado: {inv_path}')
print(f'\n✅ Fase 1 completada — process_id: {PROCESS_ID}')
spark.stop()

Inventario guardado: ../data/audit/inventory_nb01_ETL-20260623-125906-A58E4897.json

✅ Fase 1 completada — process_id: ETL-20260623-125906-A58E4897
